# IBL Neuropixels Quick Start

This notebook is a short introduction to accessing and analysing the IBL Neuropixels recordings dataset.

https://docs.internationalbrainlab.org/notebooks_external/2025_data_release_brainwidemap.html

This includes how to:

1. Connect to the IBL public database using ONE.
2. Search for Neuropixels probe insertions in a particular brain region.
3. Download spike-sorted data.
4. Explore spikes, clusters and anatomical regions.
5. Load and analyse behavioural trial data (spikes rasters and PSTHSs).

### Key IBL concepts

**Session / EID**: an experimental session, identified by an experiment ID (eid).

**Probe insertion / PID**: a particular Neuropixels probe insertion, identified by a probe insertion ID (pid).

**ALF**: the IBL Alyx File convention used to organise processed data.

**Cluster**: a unit produced by spike sorting.

**Allen Atlas acronym**: the anatomical brain-region label associated with a channel/cluster.

**ONE**: the Open Neurophysiology Environment API used to search and download IBL data.

## 1. Connect to the IBL public database using ONE.

### Import IBL libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from one.api import ONE
from brainbox.io.one import SpikeSortingLoader
from iblatlas.atlas import AllenAtlas

### Setup IBL data directory

ONE downloads data to a local cache.

It is useful to keep all IBL data for the course in one directory.

You can change this to another disk if you have limited storage.

In [ ]:
PATH_ONE = Path.home() / "IBL"

PATH_ONE.mkdir(parents=True, exist_ok=True)

print(f"IBL data will be cached in:\n{PATH_ONE}")

Connect to OpenAlyx to load data using ONE

**NOTE**: ONE API has 2 modes: Online/Offline (remote/local)

- ‘local’ mode: uses the local cache tables and not connect to Alyx to load/search data. Does not need stable internet connection

- ‘remote’ mode: spikes the local cache tables and instead query the remote database

For more info: https://int-brain-lab.github.io/ONE/notebooks/one_modes

In [ ]:
OPENALYX_URL = "https://openalyx.internationalbrainlab.org"
mode = "remote" # local/remote

one = ONE(base_url=OPENALYX_URL, mode=mode, cache_dir=PATH_ONE)
print(one.search_terms())

For detailed information on how to search with ONE:

https://int-brain-lab.github.io/ONE/notebooks/one_search/one_search.html

## 2. Search for Neuropixels probe insertions in a particular brain region.

Test search for probe insertions passing through a perticular brain region e.g. Primary visual cortex (VISp).

The project identifier for the IBL brain-wide map is: 'ibl_neuropixel_brainwide_01'

**Note:** The `search_insertions` method is only available in remote mode.

In [ ]:
atlas_acronym = "VISp"
project = "ibl_neuropixel_brainwide_01"

_, insertions = one.search_insertions(
    atlas_acronym=atlas_acronym,
    query_type="remote",
    project=project,
    datasets="spikes.times.npy",
    details=True,
)

print(f"{len(insertions)} probe insertions found passing through {atlas_acronym}")

### Extract probe IDs and session IDs

PID = probe insertion

EID = recording session containing that probe insertion

In [ ]:
pids = [ins["id"] for ins in insertions]
eids = [ins["session"] for ins in insertions]

print(f"Number of PIDs: {len(pids)}")
print(f"Number of EIDs: {len(eids)}")

print("\nExample PID:")
print(pids[0])

print("\nExample EID:")
print(eids[0])

## 3. Download spike-sorted data.

Init spike-sorting loader for a single probe recording

In [ ]:
pid = "50ebb677-e4a3-4421-b74f-1997a9cd1ad1" # example pid

print(f"Selected probe insertion:\n{pid}")

ba = AllenAtlas()

ssl = SpikeSortingLoader(pid=pid, one=one, atlas=ba)

print(f"Probe name: {ssl.pname}")
print(f"Session ID: {ssl.eid}")

Download the pykilosort spike-sorting collection.

**Note:** This may take some time the first time it is run.

In [ ]:
ssl.download_spike_sorting(collection=f"alf/{ssl.pname}/pykilosort")

print("Spike-sorting download complete.")

## 4. Explore spikes, clusters and anatomical regions.

### Load the spikesorted data

In [ ]:
SPIKESORT_REVISION = "2024-05-06"

spikes, clusters, channels = ssl.load_spike_sorting(revision=SPIKESORT_REVISION)
clusters = ssl.merge_clusters(spikes, clusters, channels)

print("Spike-sorted data loaded.")
print(f"Number of spikes: {len(spikes['times']):,}")
print(f"Number of clusters: {len(clusters['cluster_id']):,}")
print(f"Number of channels: {len(channels['localCoordinates']):,}")

Then extract basic spike variables

In [ ]:
cluster_ids = np.asarray(clusters["cluster_id"])
cluster_regions = np.asarray(clusters["acronym"])

spike_times = np.asarray(spikes["times"])
spike_clusters = np.asarray(spikes["clusters"])

spike_amps = np.asarray(spikes["amps"])
spike_depths = np.asarray(spikes["depths"])

print(f"Spike times:    {spike_times.shape}")
print(f"Spike clusters: {spike_clusters.shape}")
print(f"Spike amps:     {spike_amps.shape}")
print(f"Spike depths:   {spike_depths.shape}")

### Find clusters in a particular brain region

In [ ]:
filter_region = "VISp5"

cids_in_region = cluster_ids[cluster_regions == filter_region]

print(f"{len(cids_in_region)} clusters found in "f"{filter_region}")
print(f"Cluster IDs: {cids_in_region}")

### Analysing single neuron spikes

In [ ]:
cid = int(cids_in_region[0])

cid_spike_times = spike_times[spike_clusters == cid]
cid_spike_amplitudes = spike_amps[spike_clusters == cid]

print(f"Selected cluster: {cid} in {filter_region}")
print(f"Number of spikes: {len(cid_spike_times):,}")

Plot a 10-second snippet of the recording.

In [ ]:
t_start = 10
t_end = 20

interval_mask = (cid_spike_times >= t_start) & (cid_spike_times < t_end)

times = cid_spike_times[interval_mask]
amps = cid_spike_amplitudes[interval_mask]

fig, ax = plt.subplots(figsize=(10, 4))

ax.scatter(times, amps, s=8, color="black", alpha=0.7)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Spike amplitude")
ax.set_title(f"Cluster {cid} in {filter_region}: {t_start}–{t_end} s")

plt.show()


## 5. Load and analyse behavioural trial data.

IBL stores task-related behavioural variables in the trials ALF object.

These can be loaded via one.load_object(eid, "trials") for loading trial data.

https://docs.internationalbrainlab.org/notebooks_external/loading_trials_data.html

In [ ]:
eid = one.pid2eid(pid)[0]

trials = one.load_object(eid, 'trials', collection='alf')

print(trials.keys())

### Align spikes to a behavioural event

What does this neuron's firing look like around a particular behavioural event?

For example, consider activity 500 ms before to 500 ms after feedback.

In [ ]:
feedback_times = np.asarray(trials["feedback_times"])
window = (-0.5, 0.5) # +/- 500 ms

valid_feedback = np.isfinite(feedback_times) # only valid trials
event_times = feedback_times[valid_feedback]

trial_aligned_spikes = []

for event_time in event_times:
    start, end = event_time + window[0], event_time + window[1]
    i_start = np.searchsorted(cid_spike_times, start, side="left")
    i_end = np.searchsorted(cid_spike_times, end, side="left")
    trial_aligned_spikes.append(cid_spike_times[i_start:i_end] - event_time)


Plot spike raster

- Each row represents one trial.
- Each vertical line represents one spike.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for trial_idx, spike_times_trial in enumerate(trial_aligned_spikes):
    ax.vlines(spike_times_trial, trial_idx + 0.5, trial_idx + 1.5, color="black", linewidth=0.7)

ax.axvline(0,color="red",linestyle="--",linewidth=1,label="feedback")

ax.set_xlim(window)
ax.set_ylim(0.5, len(trial_aligned_spikes) + 0.5)

ax.set_xlabel("Time from feedback (s)")
ax.set_ylabel("Trial")

ax.set_title(
    f"Spike raster\n"
    f"cluster {cid}, {filter_region}"
)

ax.legend()

plt.show()

### Compute a PSTH

Average firing rate around an event.

In [ ]:
bin_size = 0.01  # 10 ms

bins = np.arange(window[0], window[1] + bin_size, bin_size)

counts = np.zeros(len(bins) - 1)

for spike_times_trial in trial_aligned_spikes:
    trial_counts, _ = np.histogram(spike_times_trial, bins=bins)
    counts += trial_counts

n_trials = len(trial_aligned_spikes)

firing_rate = counts / (n_trials * bin_size)

bin_centers = (bins[:-1] + bins[1:]) / 2

Plot the PSTH

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(bin_centers, firing_rate, width=bin_size, color="lightgrey", edgecolor="none")

ax.axvline(0, color="red", linestyle="--", linewidth=1)

ax.set_xlabel("Time from feedback (s)")
ax.set_ylabel("Firing rate (Hz)")

ax.set_title(f"PSTH\ncluster {cid}, {filter_region}")

plt.show()


### Comparing trial conditions

For example, correct vs incorrect trials.

+1 = correct

-1 = incorrect

In [ ]:
feedback_type = np.asarray(trials["feedbackType"])

print(f"Correct trials:   {np.sum(feedback_type == 1)}")
print(f"Incorrect trials: {np.sum(feedback_type == -1)}")

### *Exercise*: make separate PSTHs for correct and incorrect trials

In [ ]:
# Input code here